# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/co2_emissions.csv')

# Filter to Asia only
asia_df = df[df['Region'] == 'Asia']

# Pick highlight country — China tells the most dramatic story
HIGHLIGHT = 'China'
HIGHLIGHT_COLOR = '#E63946'
GREY = '#DDDDDD'

fig = go.Figure()

# Draw all non-highlighted countries first (behind)
for country, group in asia_df.groupby('Country'):
    if country == HIGHLIGHT:
        continue
    fig.add_trace(go.Scatter(
        x=group['Year'],
        y=group['CO2_Mt'],
        mode='lines',
        line=dict(color=GREY, width=1),
        hovertemplate=f'<b>{country}</b><br>Year: %{{x}}<br>CO2: %{{y:.1f}} Mt<extra></extra>',
        showlegend=False
    ))

# Draw highlighted country on top
hi = asia_df[asia_df['Country'] == HIGHLIGHT].sort_values('Year')
last = hi.iloc[-1]

fig.add_trace(go.Scatter(
    x=hi['Year'],
    y=hi['CO2_Mt'],
    mode='lines',
    line=dict(color=HIGHLIGHT_COLOR, width=3),
    hovertemplate=f'<b>{HIGHLIGHT}</b><br>Year: %{{x}}<br>CO2: %{{y:.1f}} Mt<extra></extra>',
    showlegend=False
))

# Direct label at end of line
fig.add_annotation(
    x=last['Year'],
    y=last['CO2_Mt'],
    text=f"  <b>{HIGHLIGHT}</b><br>  {last['CO2_Mt']:.0f} Mt",
    showarrow=False,
    xanchor='left',
    font=dict(color=HIGHLIGHT_COLOR, size=13)
)

fig.update_layout(
    title=dict(
        text="<b>China's CO2 Surge Dwarfs the Rest of Asia</b><br>"
             "<sup>CO2 emissions (Mt) for all Asian countries, 2000–2022</sup>",
        font=dict(size=18),
        x=0.02, xanchor='left'
    ),
    xaxis=dict(title='', showgrid=False, tickfont=dict(size=12)),
    yaxis=dict(title='CO2 Emissions (Mt)', showgrid=True,
               gridcolor='#F0F0F0', zeroline=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(l=60, r=120, t=90, b=50),
    hovermode='x unified',
    width=900, height=540
)

fig.write_html('co2_asia_highlighted.html')
fig.show()


---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [7]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go
import numpy as np

df = pd.read_csv('../data/co2_emissions.csv')

selected = ['China', 'India', 'United States', 'Germany', 'Russia', 'South Korea']

sg = df.loc[df['Country'].isin(selected) & df['Year'].isin([2000, 2022])].copy()

pivot = sg.pivot(index='Country', columns='Year', values='CO2_Mt').reset_index()
pivot.columns.name = None
pivot.columns = ['Country', 'val_2000', 'val_2022']
pivot['increased'] = pivot['val_2022'] > pivot['val_2000']
pivot['change']    = pivot['val_2022'] - pivot['val_2000']

COLOR_UP   = '#E63946'   # red   — increased
COLOR_DOWN = '#2A9D8F'   # green — decreased

# ── Label collision resolver ──────────────────────────────────────────────────
def resolve_overlaps(values, min_gap):
    pos   = np.array(values, dtype=float)
    order = np.argsort(pos)
    pos   = pos[order]

    for _ in range(300):
        moved = False
        for i in range(len(pos) - 1):
            gap = pos[i + 1] - pos[i]
            if gap < min_gap:
                push = (min_gap - gap) / 2
                pos[i]     -= push
                pos[i + 1] += push
                moved = True
        if not moved:
            break

    result = np.empty_like(pos)
    result[order] = pos
    return result

MIN_GAP = 120   # Mt — wider gap needed given China/India's large values

left_nudged  = resolve_overlaps(pivot['val_2000'].tolist(), MIN_GAP)
right_nudged = resolve_overlaps(pivot['val_2022'].tolist(), MIN_GAP)
# ─────────────────────────────────────────────────────────────────────────────

fig = go.Figure()

for i, (_, row) in enumerate(pivot.iterrows()):
    color   = COLOR_UP if row['increased'] else COLOR_DOWN
    country = row['Country']

    # Slope line
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row['val_2000'], row['val_2022']],
        mode='lines+markers',
        line=dict(color=color, width=2.5),
        marker=dict(size=8, color=color),
        hovertemplate=f"<b>{country}</b><br>2000: {row['val_2000']:.0f} Mt<br>2022: {row['val_2022']:.0f} Mt<extra></extra>",
        showlegend=False
    ))

    left_moved  = bool(abs(left_nudged[i]  - row['val_2000']) > 0.01)
    right_moved = bool(abs(right_nudged[i] - row['val_2022']) > 0.01)

    # Left label (2000)
    fig.add_annotation(
        x=2000, y=float(left_nudged[i]),
        ax=2000, ay=float(row['val_2000']),
        text=f"<b>{country}</b>  {row['val_2000']:.0f}",
        showarrow=left_moved,
        arrowhead=0, arrowcolor=color, arrowwidth=1,
        xanchor='right', font=dict(color=color, size=12, family='Arial')
    )

    # Right label (2022)
    arrow = "▲" if row['increased'] else "▼"
    fig.add_annotation(
        x=2022, y=float(right_nudged[i]),
        ax=2022, ay=float(row['val_2022']),
        text=f"{row['val_2022']:.0f}  {arrow} <b>{country}</b>",
        showarrow=right_moved,
        arrowhead=0, arrowcolor=color, arrowwidth=1,
        xanchor='left', font=dict(color=color, size=12, family='Arial')
    )

fig.update_layout(
    title=dict(
        text=f"<b>China and India dwarf all others — CO2 emissions 2000 vs 2022</b><br>"
             f"<sup><span style='color:{COLOR_UP}'>▲ Increased since 2000</span> · "
             f"<span style='color:{COLOR_DOWN}'>▼ Decreased since 2000</span> · values in Mt</sup>",
        font=dict(size=16, family='Arial'),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        tickvals=[2000, 2022],
        ticktext=['<b>2000</b>', '<b>2022</b>'],
        tickfont=dict(size=14, family='Arial'),
        showgrid=False, zeroline=False,
        range=[1990, 2032]
    ),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=180, r=180, t=90, b=50),
    height=600, width=900,
    hovermode='closest'
)

fig.write_html('co2_slopegraph_countries.html')
fig.show()